In [ ]:
VCF = "common_merged.vcf.gz"
OUT_PREFIX = "pca_common_merged"
PLINK2 = "conda run -n plink_env plink2"
SAMPLE_META = None
N_PCS = 10
WINDOW_KB = 200
STEP_SNPS = 50
R2_THRESHOLD = 0.2
FIGSIZE = (8, 6)
POINT_SIZE = 36
ALPHA = 0.85
SAVE_DPI = 300


In [ ]:
import os
from pathlib import Path
print("Current working directory:", os.getcwd())
print("VCF exists:", os.path.exists(VCF))
print("VCF path:", Path(VCF).resolve() if os.path.exists(VCF) else "NOT FOUND")


In [ ]:
import subprocess
cmd = f"{PLINK2} --version"
print("Running:", cmd)
subprocess.run(cmd, shell=True, check=True)


In [ ]:
import subprocess
def run_cmd(cmd: str):
    print("\nRunning command:")
    print(cmd)
    completed = subprocess.run(cmd, shell=True, check=True)
    print("Done.")
    return completed


In [ ]:
cmd = (
    f"{PLINK2} "
    f"--vcf {VCF} "
    f"--double-id "
    f"--allow-extra-chr "
    f"--chr 1-22 "
    f"--snps-only just-acgt "
    f"--max-alleles 2 "
    f"--set-all-var-ids @:#:$r:$a "
    f"--new-id-max-allele-len 100 "
    f"--make-bed "
    f"--out {OUT_PREFIX}"
)
run_cmd(cmd)


In [ ]:
import pandas as pd
bim = pd.read_csv(f"{OUT_PREFIX}.bim", sep=r"\s+", header=None)
bim.columns = ["CHR", "ID", "CM", "POS", "A1", "A2"]
print("Total variants:", len(bim))
print("Unique IDs:", bim["ID"].nunique())
print("Duplicated IDs:", bim["ID"].duplicated().sum())
bim[bim["ID"].duplicated(keep=False)].head(20)


In [ ]:
DEDUP_PREFIX = f"{OUT_PREFIX}_dedup"
cmd = (
    f"{PLINK2} "
    f"--bfile {OUT_PREFIX} "
    f"--rm-dup force-first "
    f"--make-bed "
    f"--out {DEDUP_PREFIX}"
)
run_cmd(cmd)


In [ ]:
cmd = (
    f"{PLINK2} "
    f"--bfile {DEDUP_PREFIX} "
    f"--indep-pairwise {WINDOW_KB} {STEP_SNPS} {R2_THRESHOLD} "
    f"--out {DEDUP_PREFIX}"
)
run_cmd(cmd)


In [ ]:
cmd = (
    f"{PLINK2} "
    f"--bfile {DEDUP_PREFIX} "
    f"--extract {DEDUP_PREFIX}.prune.in "
    f"--pca {N_PCS} "
    f"--out {DEDUP_PREFIX}"
)
run_cmd(cmd)


In [ ]:
expected_files = [
    f"{DEDUP_PREFIX}.bed",
    f"{DEDUP_PREFIX}.bim",
    f"{DEDUP_PREFIX}.fam",
    f"{DEDUP_PREFIX}.prune.in",
    f"{DEDUP_PREFIX}.eigenvec",
    f"{DEDUP_PREFIX}.eigenval",
]
for fn in expected_files:
    print(f"{fn}: {'OK' if os.path.exists(fn) else 'MISSING'}")


In [ ]:
import pandas as pd
eigenvec_path = f"{DEDUP_PREFIX}.eigenvec"
eigenval_path = f"{DEDUP_PREFIX}.eigenval"
eigenvec = pd.read_csv(eigenvec_path, sep="\t")
eigenval = pd.read_csv(eigenval_path, header=None, names=["eigenvalue"])
print("Eigenvec columns:", eigenvec.columns.tolist())
print("N samples:", len(eigenvec))
print()
display(eigenvec.head())
display(eigenval.head(10))


In [ ]:
eigenval["variance_explained_percent"] = (
    eigenval["eigenvalue"] / eigenval["eigenvalue"].sum() * 100
)
eigenval.head(10)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
EIGENVEC = "pca_common_merged_dedup.eigenvec"
EIGENVAL = "pca_common_merged_dedup.eigenval"
POP_FILE = "pop_merged.tsv"
OUT_FIG = "pca_plot_custom.png"
eig = pd.read_csv(EIGENVEC, sep=r"\s+")
cols = list(eig.columns)
cols[0] = "FID"
cols[1] = "IID"
eig.columns = cols
pc_names = [c for c in eig.columns if str(c).startswith("PC")]
if len(pc_names) < 2:
    raise ValueError("PC columns not found in eigenvec")
eig = eig[["IID"] + pc_names].copy()
for c in pc_names:
    eig[c] = pd.to_numeric(eig[c], errors="coerce")
eig = eig.dropna(subset=["IID", pc_names[0], pc_names[1]]).copy()
eig = eig.rename(columns={pc_names[0]: "PC1", pc_names[1]: "PC2"})
evals = pd.read_csv(EIGENVAL, header=None)
evals = pd.to_numeric(evals.iloc[:, 0], errors="coerce").dropna()
var_exp = evals / evals.sum() * 100
pop = pd.read_csv(POP_FILE, sep="\t")
if list(pop.columns[:2]) != ["sample_id", "population"]:
    pop = pop.iloc[:, :2].copy()
    pop.columns = ["sample_id", "population"]
pop["sample_id"] = pop["sample_id"].astype(str).str.strip()
pop["population"] = pop["population"].astype(str).str.strip()
eig["IID"] = eig["IID"].astype(str).str.strip()
df = eig.merge(pop, left_on="IID", right_on="sample_id", how="left")
df = df[~df["population"].isin(["Unknown", "No Tag", "Unknown_Behar"])].copy()
df = df[df["population"].notna()].copy()
COLORS = {
    "Ashkenazi Jews": "#F08A8A",
    "Bukharan Jews": "#A8A6E5",
    "Mountain Jews": "#F0B36E",
    "Georgian Jews": "#78C96F",
    "Kurdistani Jews": "#D08AE8",
    "Iranian Jews": "#7FC8C6",
}
BEHAR_MAP = {
    "Ashkenazi_Jews_Behar": ("Ashkenazi Jews", "Ashkenazi Jews Behar"),
    "Azerbaijani_Jews_Behar": ("Mountain Jews", "Mountain Jews Behar"),
    "Mountain_Jews_Behar": ("Mountain Jews", "Mountain Jews Behar"),
    "Georgian_Jews_Behar": ("Georgian Jews", "Georgian Jews Behar"),
    "Iranian_Jews_Behar": ("Iranian Jews", "Iranian Jews Behar"),
    "Uzbeks_Behar": ("Bukharan Jews", "Bukharan Jews Behar"),
    "Uzbekistan_Jew_Behar": ("Bukharan Jews", "Bukharan Jews Behar"),
    "Iraqi_Jews_Behar": ("Kurdistani Jews", "Kurdistani Jews Behar"),
}
MY_GROUPS = [
    "Ashkenazi Jews",
    "Bukharan Jews",
    "Mountain Jews",
    "Georgian Jews",
    "Kurdistani Jews",
]
def get_plot_group(pop_name):
    if pop_name in MY_GROUPS:
        return pop_name
    if pop_name in BEHAR_MAP:
        return BEHAR_MAP[pop_name][1]
    return "Other"
df["plot_group"] = df["population"].apply(get_plot_group)
keep_groups = MY_GROUPS + sorted(set(v[1] for v in BEHAR_MAP.values())) + ["Other"]
df = df[df["plot_group"].isin(keep_groups)].copy()
plt.figure(figsize=(10, 8), dpi=200)
other = df[df["plot_group"] == "Other"]
if len(other) > 0:
    plt.scatter(
        other["PC1"],
        other["PC2"],
        s=20,
        c="lightgray",
        alpha=0.6,
        marker="o",
        edgecolors="none",
        label="Other",
        zorder=1
    )
for group in MY_GROUPS:
    sub = df[df["plot_group"] == group]
    if len(sub) == 0:
        continue
    plt.scatter(
        sub["PC1"],
        sub["PC2"],
        s=38,
        c=COLORS[group],
        alpha=0.95,
        marker="o",
        edgecolors="black",
        linewidths=0.3,
        label=group,
        zorder=2
    )
seen = set()
for behar_name, (base_group, label_name) in BEHAR_MAP.items():
    sub = df[df["population"] == behar_name]
    if len(sub) == 0:
        continue
    label = label_name if label_name not in seen else None
    seen.add(label_name)
    plt.scatter(
        sub["PC1"],
        sub["PC2"],
        s=70,
        c=COLORS[base_group],
        alpha=1.0,
        marker="x",
        linewidths=1.6,
        label=label,
        zorder=3
    )
xlab = f"PC1 ({var_exp.iloc[0]:.2f}%)" if len(var_exp) > 0 else "PC1"
ylab = f"PC2 ({var_exp.iloc[1]:.2f}%)" if len(var_exp) > 1 else "PC2"
plt.xlabel(xlab)
plt.ylabel(ylab)
plt.title("PCA")
plt.grid(False)
handles, labels = plt.gca().get_legend_handles_labels()
uniq = {}
for h, l in zip(handles, labels):
    if l not in uniq and l is not None:
        uniq[l] = h
plt.legend(
    uniq.values(),
    uniq.keys(),
    loc="best",
    frameon=True,
    fontsize=9
)
plt.tight_layout()
plt.savefig(OUT_FIG, bbox_inches="tight")
plt.show()
print("Samples in plot:", len(df))
print()
print("My samples:")
for g in MY_GROUPS:
    print(g, (df["plot_group"] == g).sum())
print()
print("Behar refs:")
for behar_name, (_, label_name) in BEHAR_MAP.items():
    n = (df["population"] == behar_name).sum()
    if n > 0:
        print(label_name, n)
print()
print(df[["IID", "PC1", "PC2", "population"]].head())
print()
print(df[["PC1", "PC2"]].dtypes)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
EIGENVEC = "pca_common_merged_dedup.eigenvec"
EIGENVAL = "pca_common_merged_dedup.eigenval"
POP_FILE = "pop_merged.tsv"
OUT_FIG = "pca_plot_custom.png"
eig = pd.read_csv(EIGENVEC, sep=r"\s+")
cols = list(eig.columns)
cols[0] = "FID"
cols[1] = "IID"
eig.columns = cols
pc_names = [c for c in eig.columns if str(c).startswith("PC")]
if len(pc_names) < 2:
    raise ValueError("PC columns not found in eigenvec")
eig = eig[["IID"] + pc_names].copy()
for c in pc_names:
    eig[c] = pd.to_numeric(eig[c], errors="coerce")
eig = eig.dropna(subset=["IID", pc_names[0], pc_names[1]]).copy()
eig = eig.rename(columns={pc_names[0]: "PC1", pc_names[1]: "PC2"})
evals = pd.read_csv(EIGENVAL, header=None)
evals = pd.to_numeric(evals.iloc[:, 0], errors="coerce").dropna()
var_exp = evals / evals.sum() * 100
pop = pd.read_csv(POP_FILE, sep="\t")
if list(pop.columns[:2]) != ["sample_id", "population"]:
    pop = pop.iloc[:, :2].copy()
    pop.columns = ["sample_id", "population"]
pop["sample_id"] = pop["sample_id"].astype(str).str.strip()
pop["population"] = pop["population"].astype(str).str.strip()
eig["IID"] = eig["IID"].astype(str).str.strip()
df = eig.merge(pop, left_on="IID", right_on="sample_id", how="left")
df = df[~df["population"].isin(["Unknown", "No Tag", "Unknown_Behar"])].copy()
df = df[df["population"].notna()].copy()
COLORS = {
    "Ashkenazi Jews": "#F08A8A",
    "Bukharan Jews": "#A8A6E5",
    "Mountain Jews": "#F0B36E",
    "Georgian Jews": "#78C96F",
    "Kurdistani Jews": "#D08AE8",
    "Iranian Jews": "#7FC8C6",
}
BEHAR_MAP = {
    "Ashkenazi_Jews_Behar": ("Ashkenazi Jews", "Ashkenazi Jews Behar"),
    "Azerbaijani_Jews_Behar": ("Mountain Jews", "Mountain Jews Behar"),
    "Mountain_Jews_Behar": ("Mountain Jews", "Mountain Jews Behar"),
    "Georgian_Jews_Behar": ("Georgian Jews", "Georgian Jews Behar"),
    "Iranian_Jews_Behar": ("Iranian Jews", "Iranian Jews Behar"),
    "Uzbeks_Behar": ("Bukharan Jews", "Bukharan Jews Behar"),
    "Uzbekistan_Jew_Behar": ("Bukharan Jews", "Bukharan Jews Behar"),
    "Iraqi_Jews_Behar": ("Kurdistani Jews", "Kurdistani Jews Behar"),
}
MY_GROUPS = [
    "Ashkenazi Jews",
    "Bukharan Jews",
    "Mountain Jews",
    "Georgian Jews",
    "Kurdistani Jews",
]
def get_plot_group(pop_name):
    if pop_name in MY_GROUPS:
        return pop_name
    if pop_name in BEHAR_MAP:
        return BEHAR_MAP[pop_name][1]
    return "Other"
df["plot_group"] = df["population"].apply(get_plot_group)
keep_groups = MY_GROUPS + sorted(set(v[1] for v in BEHAR_MAP.values())) + ["Other"]
df = df[df["plot_group"].isin(keep_groups)].copy()
plt.figure(figsize=(10, 8), dpi=200)
other = df[df["plot_group"] == "Other"]
if len(other) > 0:
    plt.scatter(
        other["PC1"],
        other["PC2"],
        s=20,
        c="lightgray",
        alpha=0.6,
        marker="o",
        edgecolors="none",
        label="Other",
        zorder=1
    )
for group in MY_GROUPS:
    sub = df[df["plot_group"] == group]
    if len(sub) == 0:
        continue
    plt.scatter(
        sub["PC1"],
        sub["PC2"],
        s=38,
        c=COLORS[group],
        alpha=0.95,
        marker="o",
        edgecolors="black",
        linewidths=0.3,
        label=group,
        zorder=2
    )
seen = set()
for behar_name, (base_group, label_name) in BEHAR_MAP.items():
    sub = df[df["population"] == behar_name]
    if len(sub) == 0:
        continue
    label = label_name if label_name not in seen else None
    seen.add(label_name)
    plt.scatter(
        sub["PC1"],
        sub["PC2"],
        s=70,
        c=COLORS[base_group],
        alpha=1.0,
        marker="x",
        linewidths=1.6,
        label=label,
        zorder=3
    )
xlab = f"PC1 ({var_exp.iloc[0]:.2f}%)" if len(var_exp) > 0 else "PC1"
ylab = f"PC2 ({var_exp.iloc[1]:.2f}%)" if len(var_exp) > 1 else "PC2"
plt.xlabel(xlab)
plt.ylabel(ylab)
plt.title("PCA")
plt.grid(False)
handles, labels = plt.gca().get_legend_handles_labels()
uniq = {}
for h, l in zip(handles, labels):
    if l not in uniq and l is not None:
        uniq[l] = h
ax = plt.gca()
axins = inset_axes(
    ax,
    width="35%",
    height="35%",
    loc="upper left",
    borderpad=2
)
axins.scatter(
    other["PC1"],
    other["PC2"],
    s=10,
    c="lightgray",
    alpha=0.6
)
for group in MY_GROUPS:
    sub = df[df["plot_group"] == group]
    if len(sub) == 0:
        continue
    axins.scatter(
        sub["PC1"],
        sub["PC2"],
        s=25,
        c=COLORS[group],
        marker="o",
        edgecolors="black",
        linewidths=0.3
    )
for behar_name, (base_group, label_name) in BEHAR_MAP.items():
    sub = df[df["population"] == behar_name]
    if len(sub) == 0:
        continue
    axins.scatter(
        sub["PC1"],
        sub["PC2"],
        s=40,
        c=COLORS[base_group],
        marker="x",
        linewidths=1.2
    )
axins.set_xlim(-0.02, 0.05)
axins.set_ylim(-0.05, 0.02)
axins.set_xticks([])
axins.set_yticks([])
from mpl_toolkits.axes_grid1.inset_locator import mark_inset
mark_inset(ax, axins, loc1=2, loc2=4, fc="none", ec="gray", lw=1)
plt.legend(
    uniq.values(),
    uniq.keys(),
    loc="center left",
    bbox_to_anchor=(-0.22, 0.5),
    frameon=True,
    fontsize=9
)
plt.tight_layout()
plt.savefig(OUT_FIG, bbox_inches="tight")
plt.show()
print("Samples in plot:", len(df))
print()
print("My samples:")
for g in MY_GROUPS:
    print(g, (df["plot_group"] == g).sum())
print()
print("Behar refs:")
for behar_name, (_, label_name) in BEHAR_MAP.items():
    n = (df["population"] == behar_name).sum()
    if n > 0:
        print(label_name, n)
print()
print(df[["IID", "PC1", "PC2", "population"]].head())
print()
print(df[["PC1", "PC2"]].dtypes)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset
EIGENVEC = "pca_common_merged_dedup.eigenvec"
EIGENVAL = "pca_common_merged_dedup.eigenval"
POP_FILE = "pop_merged.tsv"
OUT_FIG = "pca_plot_custom.png"
eig = pd.read_csv(EIGENVEC, sep=r"\s+")
cols = list(eig.columns)
cols[0] = "FID"
cols[1] = "IID"
eig.columns = cols
pc_names = [c for c in eig.columns if str(c).startswith("PC")]
if len(pc_names) < 2:
    raise ValueError("PC columns not found in eigenvec")
eig = eig[["IID"] + pc_names].copy()
for c in pc_names:
    eig[c] = pd.to_numeric(eig[c], errors="coerce")
eig["IID"] = eig["IID"].astype(str).str.strip()
eig = eig.dropna(subset=["PC1", "PC2"]).copy()
evals = pd.read_csv(EIGENVAL, header=None)
evals = pd.to_numeric(evals.iloc[:, 0], errors="coerce").dropna()
var_exp = evals / evals.sum() * 100
pop = pd.read_csv(POP_FILE, sep="\t")
if list(pop.columns[:2]) != ["sample_id", "population"]:
    pop = pop.iloc[:, :2].copy()
    pop.columns = ["sample_id", "population"]
pop["sample_id"] = pop["sample_id"].astype(str).str.strip()
pop["population"] = pop["population"].astype(str).str.strip()
df = eig.merge(pop, left_on="IID", right_on="sample_id", how="left")
df = df[df["population"].notna()].copy()
df = df[~df["population"].isin(["Unknown", "No Tag", "Unknown_Behar"])].copy()
COLORS = {
    "Ashkenazi Jews": "#F08A8A",
    "Bukharan Jews": "#A8A6E5",
    "Mountain Jews": "#F0B36E",
    "Georgian Jews": "#78C96F",
    "Kurdistani Jews": "#D08AE8",
    "Iranian Jews": "#7FC8C6",
}
BEHAR_MAP = {
    "Ashkenazi_Jews_Behar": ("Ashkenazi Jews", "Ashkenazi Jews Behar"),
    "Azerbaijani_Jews_Behar": ("Mountain Jews", "Mountain Jews Behar"),
    "Mountain_Jews_Behar": ("Mountain Jews", "Mountain Jews Behar"),
    "Georgian_Jews_Behar": ("Georgian Jews", "Georgian Jews Behar"),
    "Iranian_Jews_Behar": ("Iranian Jews", "Iranian Jews Behar"),
    "Uzbeks_Behar": ("Bukharan Jews", "Bukharan Jews Behar"),
    "Uzbekistan_Jew_Behar": ("Bukharan Jews", "Bukharan Jews Behar"),
    "Iraqi_Jews_Behar": ("Kurdistani Jews", "Kurdistani Jews Behar"),
}
MY_GROUPS = [
    "Ashkenazi Jews",
    "Bukharan Jews",
    "Mountain Jews",
    "Georgian Jews",
    "Kurdistani Jews",
]
def get_plot_group(pop_name):
    if pop_name in MY_GROUPS:
        return pop_name
    if pop_name in BEHAR_MAP:
        return BEHAR_MAP[pop_name][1]
    return "Other"
df["plot_group"] = df["population"].apply(get_plot_group)
keep_groups = MY_GROUPS + sorted(set(v[1] for v in BEHAR_MAP.values())) + ["Other"]
df = df[df["plot_group"].isin(keep_groups)].copy()
fig, ax = plt.subplots(figsize=(11, 8), dpi=200)
other = df[df["plot_group"] == "Other"]
if len(other) > 0:
    ax.scatter(
        other["PC1"],
        other["PC2"],
        s=14,
        c="lightgray",
        alpha=0.6,
        marker="o",
        edgecolors="none",
        label="Other",
        zorder=1
    )
for group in MY_GROUPS:
    sub = df[df["plot_group"] == group]
    if len(sub) == 0:
        continue
    ax.scatter(
        sub["PC1"],
        sub["PC2"],
        s=38,
        c=COLORS[group],
        alpha=0.95,
        marker="o",
        edgecolors="black",
        linewidths=0.3,
        label=group,
        zorder=2
    )
seen = set()
for behar_name, (base_group, label_name) in BEHAR_MAP.items():
    sub = df[df["population"] == behar_name]
    if len(sub) == 0:
        continue
    label = label_name if label_name not in seen else None
    seen.add(label_name)
    ax.scatter(
        sub["PC1"],
        sub["PC2"],
        s=70,
        c=COLORS[base_group],
        alpha=1.0,
        marker="x",
        linewidths=1.6,
        label=label,
        zorder=3
    )
axins = inset_axes(
    ax,
    width="35%",
    height="35%",
    loc="upper left",
    bbox_to_anchor=(0.02, 0.98, 1, 1),
    bbox_transform=ax.transAxes,
    borderpad=0
)
for group in MY_GROUPS:
    sub = df[df["plot_group"] == group]
    if len(sub) == 0:
        continue
    axins.scatter(
        sub["PC1"],
        sub["PC2"],
        s=28,
        c=COLORS[group],
        alpha=0.95,
        marker="o",
        edgecolors="black",
        linewidths=0.3,
        zorder=2
    )
for behar_name, (base_group, _) in BEHAR_MAP.items():
    sub = df[df["population"] == behar_name]
    if len(sub) == 0:
        continue
    axins.scatter(
        sub["PC1"],
        sub["PC2"],
        s=48,
        c=COLORS[base_group],
        alpha=1.0,
        marker="x",
        linewidths=1.4,
        zorder=3
    )
axins.set_xlim(0.005, 0.035)
axins.set_ylim(-0.03, 0.005)
axins.set_xticks([])
axins.set_yticks([])
mark_inset(ax, axins, loc1=2, loc2=4, fc="none", ec="gray", lw=1)
xlab = f"PC1 ({var_exp.iloc[0]:.2f}%)" if len(var_exp) > 0 else "PC1"
ylab = f"PC2 ({var_exp.iloc[1]:.2f}%)" if len(var_exp) > 1 else "PC2"
ax.set_xlabel(xlab)
ax.set_ylabel(ylab)
ax.set_title("PCA")
ax.grid(False)
handles, labels = ax.get_legend_handles_labels()
uniq = {}
for h, l in zip(handles, labels):
    if l not in uniq and l is not None:
        uniq[l] = h
ax.legend(
    uniq.values(),
    uniq.keys(),
    loc="center left",
    bbox_to_anchor=(-0.24, 0.5),
    frameon=True,
    fontsize=9
)
plt.tight_layout()
plt.subplots_adjust(left=0.28)
plt.savefig(OUT_FIG, bbox_inches="tight")
plt.show()
print("Samples in plot:", len(df))
print()
print("My samples:")
for g in MY_GROUPS:
    print(g, (df["plot_group"] == g).sum())
print()
print("Behar refs:")
for behar_name, (_, label_name) in BEHAR_MAP.items():
    n = (df["population"] == behar_name).sum()
    if n > 0:
        print(label_name, n)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset
EIGENVEC = "pca_common_merged_dedup.eigenvec"
EIGENVAL = "pca_common_merged_dedup.eigenval"
POP_FILE = "pop_merged.tsv"
OUT_FIG = "pca_plot_custom.png"
eig = pd.read_csv(EIGENVEC, sep=r"\s+")
cols = list(eig.columns)
cols[0] = "FID"
cols[1] = "IID"
eig.columns = cols
pc_names = [c for c in eig.columns if str(c).startswith("PC")]
if len(pc_names) < 2:
    raise ValueError("PC columns not found in eigenvec")
eig = eig[["IID"] + pc_names].copy()
for c in pc_names:
    eig[c] = pd.to_numeric(eig[c], errors="coerce")
eig["IID"] = eig["IID"].astype(str).str.strip()
eig = eig.dropna(subset=[pc_names[0], pc_names[1]]).copy()
eig = eig.rename(columns={pc_names[0]: "PC1", pc_names[1]: "PC2"})
evals = pd.read_csv(EIGENVAL, header=None)
evals = pd.to_numeric(evals.iloc[:, 0], errors="coerce").dropna()
var_exp = evals / evals.sum() * 100
pop = pd.read_csv(POP_FILE, sep="\t")
if list(pop.columns[:2]) != ["sample_id", "population"]:
    pop = pop.iloc[:, :2].copy()
    pop.columns = ["sample_id", "population"]
pop["sample_id"] = pop["sample_id"].astype(str).str.strip()
pop["population"] = pop["population"].astype(str).str.strip()
df = eig.merge(pop, left_on="IID", right_on="sample_id", how="left")
df = df[df["population"].notna()].copy()
df = df[~df["population"].isin(["Unknown", "No Tag", "Unknown_Behar"])].copy()
COLORS = {
    "Ashkenazi Jews": "#F08A8A",
    "Bukharan Jews": "#A8A6E5",
    "Mountain Jews": "#F0B36E",
    "Georgian Jews": "#78C96F",
    "Kurdistani Jews": "#D08AE8",
    "Iranian Jews": "#7FC8C6",
}
BEHAR_MAP = {
    "Ashkenazi_Jews_Behar": ("Ashkenazi Jews", "Ashkenazi Jews Behar"),
    "Azerbaijani_Jews_Behar": ("Mountain Jews", "Mountain Jews Behar"),
    "Mountain_Jews_Behar": ("Mountain Jews", "Mountain Jews Behar"),
    "Georgian_Jews_Behar": ("Georgian Jews", "Georgian Jews Behar"),
    "Iranian_Jews_Behar": ("Iranian Jews", "Iranian Jews Behar"),
    "Uzbeks_Behar": ("Bukharan Jews", "Bukharan Jews Behar"),
    "Uzbekistan_Jew_Behar": ("Bukharan Jews", "Bukharan Jews Behar"),
    "Iraqi_Jews_Behar": ("Kurdistani Jews", "Kurdistani Jews Behar"),
}
MY_GROUPS = [
    "Ashkenazi Jews",
    "Bukharan Jews",
    "Mountain Jews",
    "Georgian Jews",
    "Kurdistani Jews",
]
def get_plot_group(pop_name):
    if pop_name in MY_GROUPS:
        return pop_name
    if pop_name in BEHAR_MAP:
        return BEHAR_MAP[pop_name][1]
    return "Other"
df["plot_group"] = df["population"].apply(get_plot_group)
keep_groups = MY_GROUPS + sorted(set(v[1] for v in BEHAR_MAP.values())) + ["Other"]
df = df[df["plot_group"].isin(keep_groups)].copy()
fig, ax = plt.subplots(figsize=(12, 8), dpi=200)
other = df[df["plot_group"] == "Other"]
if len(other) > 0:
    ax.scatter(
        other["PC1"], other["PC2"],
        s=14, c="lightgray", alpha=0.6,
        marker="o", edgecolors="none",
        label="Other", zorder=1
    )
for group in MY_GROUPS:
    sub = df[df["plot_group"] == group]
    if len(sub) == 0:
        continue
    ax.scatter(
        sub["PC1"], sub["PC2"],
        s=38, c=COLORS[group], alpha=0.95,
        marker="o", edgecolors="black", linewidths=0.3,
        label=group, zorder=2
    )
seen = set()
for behar_name, (base_group, label_name) in BEHAR_MAP.items():
    sub = df[df["population"] == behar_name]
    if len(sub) == 0:
        continue
    label = label_name if label_name not in seen else None
    seen.add(label_name)
    ax.scatter(
        sub["PC1"], sub["PC2"],
        s=70, c=COLORS[base_group], alpha=1.0,
        marker="x", linewidths=1.6,
        label=label, zorder=3
    )
axins = inset_axes(
    ax,
    width="34%",
    height="34%",
    loc="upper left"
)
for group in MY_GROUPS:
    sub = df[df["plot_group"] == group]
    if len(sub) == 0:
        continue
    axins.scatter(
        sub["PC1"], sub["PC2"],
        s=24, c=COLORS[group], alpha=0.95,
        marker="o", edgecolors="black", linewidths=0.3,
        zorder=2
    )
for behar_name, (base_group, _) in BEHAR_MAP.items():
    sub = df[df["population"] == behar_name]
    if len(sub) == 0:
        continue
    axins.scatter(
        sub["PC1"], sub["PC2"],
        s=42, c=COLORS[base_group], alpha=1.0,
        marker="x", linewidths=1.3,
        zorder=3
    )
axins.set_xlim(0.008, 0.032)
axins.set_ylim(-0.030, 0.005)
axins.set_xticks([])
axins.set_yticks([])
mark_inset(ax, axins, loc1=1, loc2=3, fc="none", ec="gray", lw=0.8)
xlab = "PC1"
ylab = "PC2"
ax.set_xlabel(xlab)
ax.set_ylabel(ylab)
ax.set_title("PCA")
ax.grid(False)
handles, labels = ax.get_legend_handles_labels()
uniq = {}
for h, l in zip(handles, labels):
    if l not in uniq and l is not None:
        uniq[l] = h
ax.legend(
    uniq.values(),
    uniq.keys(),
    loc="lower left",
    frameon=True,
    fontsize=9
)
fig.subplots_adjust(left=0.22, right=0.98, top=0.94, bottom=0.1)
fig.savefig(OUT_FIG, dpi=300)
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
pca = pd.read_csv('pca_jews.eigenvec', sep=r'\s+', header=None)
n_cols = pca.shape[1]
pca.columns = ['FID', 'ID'] + [f'PC{i}' for i in range(1, n_cols - 1)]
eigvals = pd.read_csv('pca_jews.eigenval', header=None)[0]
var_explained = eigvals / eigvals.sum() * 100
groups = pd.read_csv('groupJEWS.csv')
df = pca.merge(groups, on='ID', how='left')
df = df[
    df['Population'].notna() &
    (~df['Population'].isin(['Unknown', 'No Tag']))
]
color_map = {
    'Ashkenazi Jews': '#7b2cbf',
    'Bukharan Jews': '#6ec5ff',
    'Georgian Jews': '#d62728',
    'Kurdistan Jews': '#8fd175',
    'Mountain Jews': '#f39c12'
}
plt.figure(figsize=(10, 10))
for pop, color in color_map.items():
    sub = df[df['Population'] == pop]
    if len(sub) > 0:
        plt.scatter(
            sub['PC1'],
            sub['PC2'],
            label=pop,
            color=color,
            alpha=0.7,
            s=80,
            edgecolors='k',
            linewidths=0.4
        )
plt.xlabel(f'PC1 ({var_explained.iloc[0]:.2f}%)', fontsize=18)
plt.ylabel(f'PC2 ({var_explained.iloc[1]:.2f}%)', fontsize=18)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.legend(
    loc="lower left",
    fontsize=14,
    frameon=True,
    framealpha=1,
    edgecolor="black"
)
plt.savefig('PCA_Barenbaum_lil.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
INPUT = "clusters_with_pca.tsv"
OUTFIG = "pca_GTBB_2panels_transparent.png"
FIGSIZE = (18, 8)
DPI = 300
PC1_VAR = 7.76
PC2_VAR = 6.70
PC3_VAR = 4.95
KURDISTAN_IDS = {
    "kt3432", "ga2441", "sd4612", "ig2110",
    "vo0892", "jc3367", "ps5221", "au9645"
}
POINT_SIZE = 85
POINT_ALPHA = 0.75
EDGE_COLOR = "black"
EDGE_WIDTH = 0.4
LABEL_FONTSIZE = 17
TICK_FONTSIZE = 14
LEGEND_FONTSIZE = 19
TITLE_FONTSIZE = 17
LEGEND_MARKERSIZE = 14
df = pd.read_csv(INPUT, sep="\t").copy()
required_cols = ["tubeid", "cluster1", "PC1", "PC2", "PC3"]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Column '{col}' not found. Available columns: {df.columns.tolist()}")
df["tubeid"] = df["tubeid"].astype(str).str.strip().str.lower()
KURDISTAN_IDS = {x.strip().lower() for x in KURDISTAN_IDS}
df = df[df["PC1"].notna() & df["PC2"].notna() & df["PC3"].notna()].copy()
df["sample_type"] = df["tubeid"].apply(
    lambda x: "Genotek" if x == "external" else "Curated cohort"
)
cluster_label_map = {
    "Ashkenazim": "Ashkenazi Jews",
    "Jews-Azerbaijani": "Mountain Jews",
    "Jews-Georgian": "Georgian Jews",
    "Jews-Uzbekistani": "Bukharan Jews",
}
df["plot_group"] = df["cluster1"].astype(str).str.strip().map(cluster_label_map)
df.loc[df["tubeid"].isin(KURDISTAN_IDS), "plot_group"] = "Kurdistan Jews"
df = df[df["plot_group"].notna()].copy()
color_map = {
    "Ashkenazi Jews": "#7b2cbf",
    "Bukharan Jews": "#6ec5ff",
    "Georgian Jews": "#d62728",
    "Kurdistan Jews": "#8fd175",
    "Mountain Jews": "#f39c12"
}
group_order = [
    "Ashkenazi Jews",
    "Bukharan Jews",
    "Georgian Jews",
    "Kurdistan Jews",
    "Mountain Jews",
]
present_groups = [g for g in group_order if g in df["plot_group"].unique()]
def scatter_group(ax, sub, xcol, ycol, marker, zorder=3):
    ax.scatter(
        sub[xcol],
        sub[ycol],
        s=POINT_SIZE,
        c=sub["plot_group"].map(color_map),
        marker=marker,
        alpha=POINT_ALPHA,
        edgecolors=EDGE_COLOR,
        linewidths=EDGE_WIDTH,
        zorder=zorder
    )
def plot_panel(ax, xcol, ycol, xlabel, ylabel, title):
    for grp in present_groups:
        if grp == "Kurdistan Jews":
            continue
        sub = df[df["plot_group"] == grp]
        if sub.empty:
            continue
        sub_genotek = sub[sub["sample_type"] == "Genotek"]
        sub_curated = sub[sub["sample_type"] == "Curated cohort"]
        if not sub_genotek.empty:
            scatter_group(ax, sub_genotek, xcol, ycol, marker="^", zorder=2)
        if not sub_curated.empty:
            scatter_group(ax, sub_curated, xcol, ycol, marker="o", zorder=3)
    if "Kurdistan Jews" in present_groups:
        sub = df[df["plot_group"] == "Kurdistan Jews"]
        sub_genotek = sub[sub["sample_type"] == "Genotek"]
        sub_curated = sub[sub["sample_type"] == "Curated cohort"]
        if not sub_genotek.empty:
            scatter_group(ax, sub_genotek, xcol, ycol, marker="^", zorder=10)
        if not sub_curated.empty:
            scatter_group(ax, sub_curated, xcol, ycol, marker="o", zorder=11)
    ax.set_xlabel(xlabel, fontsize=LABEL_FONTSIZE)
    ax.set_ylabel(ylabel, fontsize=LABEL_FONTSIZE)
    ax.tick_params(axis="both", labelsize=TICK_FONTSIZE)
    ax.set_title(title, fontsize=TITLE_FONTSIZE)
fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)
plot_panel(
    axes[0],
    "PC1",
    "PC2",
    f"PC1 ({PC1_VAR:.2f}%)",
    f"PC2 ({PC2_VAR:.2f}%)",
    "PC1 vs PC2"
)
plot_panel(
    axes[1],
    "PC2",
    "PC3",
    f"PC2 ({PC2_VAR:.2f}%)",
    f"PC3 ({PC3_VAR:.2f}%)",
    "PC2 vs PC3"
)
fig.patch.set_alpha(0)
for ax in axes:
    ax.set_facecolor("none")
legend_handles = []
for g in present_groups:
    legend_handles.append(
        Line2D(
            [0], [0],
            marker="o",
            color="w",
            markerfacecolor=color_map[g],
            markeredgecolor=EDGE_COLOR if g != "Other" else "none",
            markeredgewidth=EDGE_WIDTH,
            markersize=LEGEND_MARKERSIZE,
            linestyle="None",
            label=g
        )
    )
legend_handles.extend([
    Line2D(
        [0], [0],
        marker="o",
        color="black",
        markerfacecolor="white",
        markeredgecolor="black",
        markersize=LEGEND_MARKERSIZE,
        linestyle="None",
        label="Curated cohort"
    ),
    Line2D(
        [0], [0],
        marker="^",
        color="black",
        markerfacecolor="white",
        markeredgecolor="black",
        markersize=LEGEND_MARKERSIZE,
        linestyle="None",
        label="Genotek"
    )
])
fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=4,
    frameon=True,
    facecolor="white",
    edgecolor="black",
    fontsize=LEGEND_FONTSIZE,
    bbox_to_anchor=(0.5, -0.06)
)
plt.tight_layout(rect=[0, 0.11, 1, 1])
plt.savefig(
    OUTFIG,
    dpi=DPI,
    bbox_inches="tight",
    transparent=True
)
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
INPUT = "merged_renamed_sds.tsv"
OUTFIG = "pca_GTBB_2panels_pres.png"
FIGSIZE = (18, 8)
DPI = 250
PC1_VAR = 7.76
PC2_VAR = 6.70
PC3_VAR = 4.95
KURDISTAN_IDS = {
    "kt3432", "ga2441", "sd4612", "ig2110",
    "vo0892", "jc3367", "ps5221", "au9645"
}
POINT_SIZE = 80
POINT_ALPHA = 0.7
EDGE_COLOR = "black"
EDGE_WIDTH = 0.4
LABEL_FONTSIZE = 16
TICK_FONTSIZE = 13
LEGEND_FONTSIZE = 13
TITLE_FONTSIZE = 16
df = pd.read_csv(INPUT, sep="\t").copy()
df["sample_type"] = df["sample_id"].apply(
    lambda x: "Genotek" if str(x) == "EXTERNAL" else "Curated cohort"
)
cluster_label_map = {
    "Ashkenazim": "Ashkenazi Jews",
    "Jews-Azerbaijani": "Mountain Jews",
    "Jews-Georgian": "Georgian Jews",
    "Jews-Uzbekistani": "Bukharan Jews",
}
df["plot_group"] = df["cluster1"].map(cluster_label_map)
df.loc[df["sample_id"].isin(KURDISTAN_IDS), "plot_group"] = "Kurdistan Jews"
df["plot_group"] = df["plot_group"].fillna("Other")
color_map = {
    "Ashkenazi Jews": "#7b2cbf",
    "Bukharan Jews": "#6ec5ff",
    "Georgian Jews": "#d62728",
    "Kurdistan Jews": "#8fd175",
    "Mountain Jews": "#f39c12",
    "Other": "#8f8f8f"
}
df["color"] = df["plot_group"].map(color_map)
group_order = [
    "Ashkenazi Jews",
    "Bukharan Jews",
    "Georgian Jews",
    "Kurdistan Jews",
    "Mountain Jews",
    "Other"
]
present_groups = [g for g in group_order if g in df["plot_group"].unique()]
def plot_panel(ax, xcol, ycol, xlabel, ylabel, title=None):
    for grp in present_groups:
        sub = df[df["plot_group"] == grp]
        if len(sub) == 0:
            continue
        sub_genotek = sub[sub["sample_type"] == "Genotek"]
        if len(sub_genotek) > 0:
            ax.scatter(
                sub_genotek[xcol],
                sub_genotek[ycol],
                s=POINT_SIZE,
                c=sub_genotek["color"],
                marker="^",
                alpha=POINT_ALPHA,
                edgecolors=EDGE_COLOR,
                linewidths=EDGE_WIDTH
            )
        sub_curated = sub[sub["sample_type"] == "Curated cohort"]
        if len(sub_curated) > 0:
            ax.scatter(
                sub_curated[xcol],
                sub_curated[ycol],
                s=POINT_SIZE,
                c=sub_curated["color"],
                marker="o",
                alpha=POINT_ALPHA,
                edgecolors=EDGE_COLOR,
                linewidths=EDGE_WIDTH
            )
    ax.set_xlabel(xlabel, fontsize=LABEL_FONTSIZE)
    ax.set_ylabel(ylabel, fontsize=LABEL_FONTSIZE)
    ax.tick_params(axis="both", labelsize=TICK_FONTSIZE)
    if title is not None:
        ax.set_title(title, fontsize=TITLE_FONTSIZE)
fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)
plot_panel(
    axes[0],
    xcol="PC1",
    ycol="PC2",
    xlabel=f"PC1 ({PC1_VAR:.2f}%)",
    ylabel=f"PC2 ({PC2_VAR:.2f}%)",
    title="PC1 vs PC2"
)
plot_panel(
    axes[1],
    xcol="PC2",
    ycol="PC3",
    xlabel=f"PC2 ({PC2_VAR:.2f}%)",
    ylabel=f"PC3 ({PC3_VAR:.2f}%)",
    title="PC2 vs PC3"
)
legend_handles = []
for g in present_groups:
    legend_handles.append(
        Line2D(
            [0], [0],
            marker="o",
            color="w",
            markerfacecolor=color_map[g],
            markeredgecolor=EDGE_COLOR if g != "Other" else "none",
            markeredgewidth=EDGE_WIDTH,
            markersize=9,
            linestyle="None",
            label=g
        )
    )
legend_handles.extend([
    Line2D(
        [0], [0],
        marker="o",
        color="black",
        markerfacecolor="white",
        markeredgecolor="black",
        markersize=9,
        linestyle="None",
        label="Curated cohort"
    ),
    Line2D(
        [0], [0],
        marker="^",
        color="black",
        markerfacecolor="white",
        markeredgecolor="black",
        markersize=9,
        linestyle="None",
        label="Genotek"
    )
])
fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=4,
    frameon=True,
    facecolor="white",
    edgecolor="black",
    fontsize=LEGEND_FONTSIZE,
    bbox_to_anchor=(0.5, -0.02)
)
plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.savefig(OUTFIG, dpi=DPI, bbox_inches="tight")
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
INPUT = "merged_renamed_sds.tsv"
OUTFIG = "pca_GT_curated_fixed_transparent.png"
FIGSIZE = (18, 8)
DPI = 250
PC1_VAR = 7.76
PC2_VAR = 6.70
PC3_VAR = 4.95
KURDISTAN_IDS = {
    "kt3432", "ga2441", "sd4612", "ig2110",
    "vo0892", "jc3367", "ps5221", "au9645"
}
POINT_SIZE = 80
POINT_ALPHA = 0.7
EDGE_COLOR = "black"
EDGE_WIDTH = 0.4
LABEL_FONTSIZE = 16
TICK_FONTSIZE = 13
LEGEND_FONTSIZE = 17
TITLE_FONTSIZE = 16
LEGEND_MARKERSIZE = 12
df = pd.read_csv(INPUT, sep="\t").copy()
print("Columns:", df.columns.tolist())
print("N rows total:", len(df))
required_pcs = ["PC1", "PC2", "PC3"]
for col in required_pcs:
    if col not in df.columns:
        raise ValueError(f"Column '{col}' not found. Available columns: {df.columns.tolist()}")
df = df[df["PC1"].notna() & df["PC2"].notna() & df["PC3"].notna()].copy()
print("Rows with PCA coordinates:", len(df))
if "groups" not in df.columns:
    raise ValueError(f"Column 'groups' not found. Available columns: {df.columns.tolist()}")
df["sample_type"] = df["groups"].apply(
    lambda x: "Curated cohort" if pd.notna(x) else "Genotek"
)
if "cluster1" not in df.columns:
    raise ValueError(f"Column 'cluster1' not found. Available columns: {df.columns.tolist()}")
group_map = {
    "Ashkenazi_Jews": "Ashkenazi Jews",
    "Bukharan_Jews": "Bukharan Jews",
    "Georgian_Jews": "Georgian Jews",
    "Kurdistani_Jews": "Kurdistani Jews",
    "Mountain_Jews": "Mountain Jews",
}
cluster_map = {
    "Ashkenazim": "Ashkenazi Jews",
    "Jews-Uzbekistani": "Bukharan Jews",
    "Jews-Georgian": "Georgian Jews",
    "Jews-Iranian": "Iranian Jews",
    "Jews-Azerbaijani": "Mountain Jews",
    "Armenians,Hemshins": "Armenians",
    "Armenians": "Armenians",
}
df["plot_group"] = df["groups"].map(group_map)
df = df[df["plot_group"] != "Iranian Jews"].copy()
mask = df["plot_group"].isna()
df.loc[mask, "plot_group"] = df.loc[mask, "cluster1"].astype(str).str.strip().map(cluster_map)
if "sample_id" not in df.columns:
    raise ValueError(f"Column 'sample_id' not found. Available columns: {df.columns.tolist()}")
df["sample_id"] = df["sample_id"].astype(str).str.strip()
df.loc[df["sample_id"].isin(KURDISTAN_IDS), "plot_group"] = "Kurdistani Jews"
df["plot_group"] = df["plot_group"].fillna("No Tag")
df = df[df["plot_group"] != "No Tag"].copy()
df = df[df["plot_group"] != "Armenians"].copy()
print("\nCounts by plot_group:")
print(df["plot_group"].value_counts(dropna=False))
print("\nCounts by sample_type:")
print(df["sample_type"].value_counts(dropna=False))
color_map = {
    "Ashkenazi Jews": "#7b2cbf",
    "Bukharan Jews": "#6ec5ff",
    "Georgian Jews": "#d62728",
    "Kurdistani Jews": "#8fd175",
    "Mountain Jews": "#f39c12",
    "Iranian Jews": "#4daf4a",
    "Other": "#8f8f8f",
}
df["color"] = df["plot_group"].map(color_map).fillna("#8f8f8f")
group_order = [
    "Ashkenazi Jews",
    "Bukharan Jews",
    "Georgian Jews",
    "Kurdistani Jews",
    "Mountain Jews",
    "Iranian Jews",
    "Other",
]
present_groups = [g for g in group_order if g in df["plot_group"].unique()]
def plot_panel(ax, xcol, ycol, xlabel, ylabel, title=None):
    for grp in present_groups:
        sub = df[df["plot_group"] == grp]
        if sub.empty:
            continue
        sub_genotek = sub[sub["sample_type"] == "Genotek"]
        if not sub_genotek.empty:
            ax.scatter(
                sub_genotek[xcol],
                sub_genotek[ycol],
                s=POINT_SIZE,
                c=sub_genotek["color"],
                marker="^",
                alpha=POINT_ALPHA,
                edgecolors=EDGE_COLOR,
                linewidths=EDGE_WIDTH
            )
        sub_curated = sub[sub["sample_type"] == "Curated cohort"]
        if not sub_curated.empty:
            ax.scatter(
                sub_curated[xcol],
                sub_curated[ycol],
                s=POINT_SIZE,
                c=sub_curated["color"],
                marker="o",
                alpha=POINT_ALPHA,
                edgecolors=EDGE_COLOR,
                linewidths=EDGE_WIDTH
            )
    ax.set_xlabel(xlabel, fontsize=LABEL_FONTSIZE)
    ax.set_ylabel(ylabel, fontsize=LABEL_FONTSIZE)
    ax.tick_params(axis="both", labelsize=TICK_FONTSIZE)
    if title:
        ax.set_title(title, fontsize=TITLE_FONTSIZE)
fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)
plot_panel(
    axes[0],
    xcol="PC1",
    ycol="PC2",
    xlabel=f"PC1 ({PC1_VAR:.2f}%)",
    ylabel=f"PC2 ({PC2_VAR:.2f}%)",
    title="PC1 vs PC2"
)
plot_panel(
    axes[1],
    xcol="PC2",
    ycol="PC3",
    xlabel=f"PC2 ({PC2_VAR:.2f}%)",
    ylabel=f"PC3 ({PC3_VAR:.2f}%)",
    title="PC2 vs PC3"
)
fig.patch.set_alpha(0)
for ax in axes:
    ax.set_facecolor("none")
legend_handles = []
for g in present_groups:
    legend_handles.append(
        Line2D(
            [0], [0],
            marker="o",
            color="w",
            markerfacecolor=color_map[g],
            markeredgecolor=EDGE_COLOR if g != "Other" else "none",
            markeredgewidth=EDGE_WIDTH,
            markersize=LEGEND_MARKERSIZE,
            linestyle="None",
            label=g
        )
    )
legend_handles.extend([
    Line2D(
        [0], [0],
        marker="o",
        color="black",
        markerfacecolor="white",
        markeredgecolor="black",
        markersize=LEGEND_MARKERSIZE,
        linestyle="None",
        label="Curated cohort"
    ),
    Line2D(
        [0], [0],
        marker="^",
        color="black",
        markerfacecolor="white",
        markeredgecolor="black",
        markersize=LEGEND_MARKERSIZE,
        linestyle="None",
        label="Genotek"
    )
])
fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=5,
    frameon=True,
    facecolor="white",
    edgecolor="black",
    fontsize=LEGEND_FONTSIZE,
    bbox_to_anchor=(0.5, -0.04)
)
plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.savefig(
    OUTFIG,
    dpi=DPI,
    bbox_inches="tight",
    transparent=True
)
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
INPUT = "merged_renamed_sds.tsv"
OUTFIG = "pca_GT_curated_fixed_transparent.png"
FIGSIZE = (20, 8.5)
DPI = 300
PC1_VAR = 7.76
PC2_VAR = 6.70
PC3_VAR = 4.95
KURDISTAN_IDS = {
    "kt3432", "ga2441", "sd4612", "ig2110",
    "vo0892", "jc3367", "ps5221", "au9645"
}
POINT_SIZE = 95
POINT_ALPHA = 0.80
EDGE_COLOR = "black"
EDGE_WIDTH = 0.45
LABEL_FONTSIZE = 17
TICK_FONTSIZE = 14
TITLE_FONTSIZE = 17
LEGEND_FONTSIZE = 20
LEGEND_MARKERSIZE = 15
df = pd.read_csv(INPUT, sep="\t").copy()
print("Columns:", df.columns.tolist())
print("N rows total:", len(df))
required_cols = ["PC1", "PC2", "PC3", "groups", "cluster1", "sample_id"]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Column '{col}' not found. Available columns: {df.columns.tolist()}")
df = df[df["PC1"].notna() & df["PC2"].notna() & df["PC3"].notna()].copy()
print("Rows with PCA coordinates:", len(df))
df["sample_id"] = df["sample_id"].astype(str).str.strip().str.lower()
KURDISTAN_IDS = {x.strip().lower() for x in KURDISTAN_IDS}
df["sample_type"] = df["groups"].apply(
    lambda x: "Curated cohort" if pd.notna(x) and str(x).strip() != "" else "Genotek"
)
group_map = {
    "Ashkenazi_Jews": "Ashkenazi Jews",
    "Bukharan_Jews": "Bukharan Jews",
    "Georgian_Jews": "Georgian Jews",
    "Kurdistani_Jews": "Kurdistani Jews",
    "Mountain_Jews": "Mountain Jews",
}
cluster_map = {
    "Ashkenazim": "Ashkenazi Jews",
    "Jews-Uzbekistani": "Bukharan Jews",
    "Jews-Georgian": "Georgian Jews",
    "Jews-Azerbaijani": "Mountain Jews",
    "Jews-Iranian": "Iranian Jews",
    "Armenians": "Armenians",
    "Armenians,Hemshins": "Armenians",
}
df["plot_group"] = df["groups"].astype(str).str.strip().map(group_map)
df.loc[df["sample_id"].isin(KURDISTAN_IDS), "plot_group"] = "Kurdistani Jews"
mask = df["plot_group"].isna()
df.loc[mask, "plot_group"] = (
    df.loc[mask, "cluster1"]
      .astype(str)
      .str.strip()
      .map(cluster_map)
)
df["plot_group"] = df["plot_group"].fillna("No Tag")
df = df[~df["plot_group"].isin(["Armenians", "Iranian Jews", "No Tag"])].copy()
print("\nCounts by plot_group:")
print(df["plot_group"].value_counts(dropna=False))
print("\nCounts by sample_type:")
print(df["sample_type"].value_counts(dropna=False))
print("\nManual Kurdistani samples found:")
print(df[df["sample_id"].isin(KURDISTAN_IDS)][["sample_id", "plot_group", "sample_type"]])
color_map = {
    "Ashkenazi Jews": "#7b2cbf",
    "Bukharan Jews": "#6ec5ff",
    "Georgian Jews": "#d62728",
    "Kurdistani Jews": "#8fd175",
    "Mountain Jews": "#f39c12",
}
group_order = [
    "Ashkenazi Jews",
    "Bukharan Jews",
    "Georgian Jews",
    "Kurdistani Jews",
    "Mountain Jews",
]
present_groups = [g for g in group_order if g in df["plot_group"].unique()]
def draw_group(ax, sub, xcol, ycol, marker, zorder):
    ax.scatter(
        sub[xcol],
        sub[ycol],
        s=POINT_SIZE,
        c=sub["plot_group"].map(color_map),
        marker=marker,
        alpha=POINT_ALPHA,
        edgecolors=EDGE_COLOR,
        linewidths=EDGE_WIDTH,
        zorder=zorder
    )
def plot_panel(ax, xcol, ycol, xlabel, ylabel, title=None):
    for grp in present_groups:
        if grp == "Kurdistani Jews":
            continue
        sub = df[df["plot_group"] == grp]
        if sub.empty:
            continue
        sub_genotek = sub[sub["sample_type"] == "Genotek"]
        sub_curated = sub[sub["sample_type"] == "Curated cohort"]
        if not sub_genotek.empty:
            draw_group(ax, sub_genotek, xcol, ycol, marker="^", zorder=2)
        if not sub_curated.empty:
            draw_group(ax, sub_curated, xcol, ycol, marker="o", zorder=3)
    if "Kurdistani Jews" in present_groups:
        sub = df[df["plot_group"] == "Kurdistani Jews"]
        sub_genotek = sub[sub["sample_type"] == "Genotek"]
        sub_curated = sub[sub["sample_type"] == "Curated cohort"]
        if not sub_genotek.empty:
            draw_group(ax, sub_genotek, xcol, ycol, marker="^", zorder=10)
        if not sub_curated.empty:
            draw_group(ax, sub_curated, xcol, ycol, marker="o", zorder=11)
    ax.set_xlabel(xlabel, fontsize=LABEL_FONTSIZE)
    ax.set_ylabel(ylabel, fontsize=LABEL_FONTSIZE)
    ax.tick_params(axis="both", labelsize=TICK_FONTSIZE)
    if title:
        ax.set_title(title, fontsize=TITLE_FONTSIZE)
fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)
plot_panel(
    axes[0],
    xcol="PC1",
    ycol="PC2",
    xlabel=f"PC1 ({PC1_VAR:.2f}%)",
    ylabel=f"PC2 ({PC2_VAR:.2f}%)",
    title="PC1 vs PC2"
)
plot_panel(
    axes[1],
    xcol="PC2",
    ycol="PC3",
    xlabel=f"PC2 ({PC2_VAR:.2f}%)",
    ylabel=f"PC3 ({PC3_VAR:.2f}%)",
    title="PC2 vs PC3"
)
fig.patch.set_alpha(0)
for ax in axes:
    ax.set_facecolor("none")
legend_handles = []
for g in present_groups:
    legend_handles.append(
        Line2D(
            [0], [0],
            marker="o",
            color="w",
            markerfacecolor=color_map[g],
            markeredgecolor="black",
            markeredgewidth=EDGE_WIDTH,
            markersize=LEGEND_MARKERSIZE,
            linestyle="None",
            label=g
        )
    )
legend_handles.extend([
    Line2D(
        [0], [0],
        marker="o",
        color="black",
        markerfacecolor="white",
        markeredgecolor="black",
        markersize=LEGEND_MARKERSIZE,
        linestyle="None",
        label="Curated cohort"
    ),
    Line2D(
        [0], [0],
        marker="^",
        color="black",
        markerfacecolor="white",
        markeredgecolor="black",
        markersize=LEGEND_MARKERSIZE,
        linestyle="None",
        label="Genotek"
    )
])
fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=4,
    frameon=True,
    facecolor="white",
    edgecolor="black",
    fontsize=LEGEND_FONTSIZE,
    bbox_to_anchor=(0.5, -0.06)
)
plt.tight_layout(rect=[0, 0.12, 1, 1])
plt.savefig(
    OUTFIG,
    dpi=DPI,
    bbox_inches="tight",
    transparent=True
)
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
EIGENVEC = "pca_common_merged_dedup.eigenvec"
EIGENVAL = "pca_common_merged_dedup.eigenval"
POP_FILE = "pop_merged.tsv"
OUT_FIG = "PCAwBehar_2_pres.png"
eig = pd.read_csv(EIGENVEC, sep=r"\s+")
cols = list(eig.columns)
cols[0] = "FID"
cols[1] = "IID"
eig.columns = cols
pc_names = [c for c in eig.columns if str(c).startswith("PC")]
if len(pc_names) < 2:
    raise ValueError("PC columns not found in eigenvec")
eig = eig[["IID"] + pc_names].copy()
for c in pc_names:
    eig[c] = pd.to_numeric(eig[c], errors="coerce")
eig["IID"] = eig["IID"].astype(str).str.strip()
eig = eig.dropna(subset=[pc_names[0], pc_names[1]]).copy()
eig = eig.rename(columns={pc_names[0]: "PC1", pc_names[1]: "PC2"})
evals = pd.read_csv(EIGENVAL, header=None)
evals = pd.to_numeric(evals.iloc[:, 0], errors="coerce").dropna()
var_exp = evals / evals.sum() * 100
pc1_var = float(var_exp.iloc[0]) if len(var_exp) > 0 else None
pc2_var = float(var_exp.iloc[1]) if len(var_exp) > 1 else None
pop = pd.read_csv(POP_FILE, sep="\t")
if list(pop.columns[:2]) != ["sample_id", "population"]:
    pop = pop.iloc[:, :2].copy()
    pop.columns = ["sample_id", "population"]
pop["sample_id"] = pop["sample_id"].astype(str).str.strip()
pop["population"] = pop["population"].astype(str).str.strip()
df = eig.merge(pop, left_on="IID", right_on="sample_id", how="left")
df = df[df["population"].notna()].copy()
df = df[~df["population"].isin(["Unknown", "No Tag", "Unknown_Behar"])].copy()
COLORS = {
    "Ashkenazi Jews": "#7b2cbf",
    "Bukharan Jews": "#6ec5ff",
    "Georgian Jews": "#d62728",
    "Kurdistan Jews": "#8fd175",
    "Mountain Jews": "#f39c12",
    "Iranian Jews": "#1f77b4",
    "Non-Jewish": "#8f8f8f"
}
BEHAR_MAP = {
    "Ashkenazi_Jews_Behar": ("Ashkenazi Jews", "Ashkenazi Jews (Behar)"),
    "Uzbeks_Behar": ("Bukharan Jews", "Uzbekistani Jews (Behar)"),
    "Uzbekistan_Jew_Behar": ("Bukharan Jews", "Uzbekistani Jews (Behar)"),
    "Georgian_Jews_Behar": ("Georgian Jews", "Georgian Jews (Behar)"),
    "Iraqi_Jews_Behar": ("Kurdistan Jews", "Iraqi Jews (Behar)"),
    "Azerbaijani_Jews_Behar": ("Mountain Jews", "Azerbaijani Jews (Behar)"),
    "Mountain_Jews_Behar": ("Mountain Jews", "Azerbaijani Jews (Behar)"),
    "Iranian_Jews_Behar": ("Iranian Jews", "Iranian Jews (Behar)"),
}
MY_GROUPS = [
    "Ashkenazi Jews",
    "Bukharan Jews",
    "Georgian Jews",
    "Kurdistan Jews",
    "Mountain Jews",
]
REF_GROUP_ORDER = [
    "Ashkenazi Jews (Behar)",
    "Uzbekistani Jews (Behar)",
    "Georgian Jews (Behar)",
    "Iraqi Jews (Behar)",
    "Azerbaijani Jews (Behar)",
    "Iranian Jews (Behar)",
]
def get_plot_group(pop_name):
    if pop_name in MY_GROUPS:
        return pop_name
    if pop_name in BEHAR_MAP:
        return BEHAR_MAP[pop_name][1]
    return "Non-Jewish"
df["plot_group"] = df["population"].apply(get_plot_group)
keep_groups = MY_GROUPS + REF_GROUP_ORDER + ["Non-Jewish"]
df = df[df["plot_group"].isin(keep_groups)].copy()
core_jewish = df[df["plot_group"] != "Non-Jewish"].copy()
other = df[df["plot_group"] == "Non-Jewish"].copy()
fig, (ax1, ax2) = plt.subplots(
    1, 2,
    figsize=(18, 8),
    dpi=200,
    gridspec_kw={"width_ratios": [1.15, 1]}
)
if len(other) > 0:
    ax1.scatter(
        other["PC1"], other["PC2"],
        s=14,
        c=COLORS["Non-Jewish"],
        alpha=0.5,
        marker="o",
        edgecolors="none",
        label="Non-Jewish",
        zorder=1
    )
for group in MY_GROUPS:
    sub = df[df["plot_group"] == group]
    if len(sub) == 0:
        continue
    ax1.scatter(
        sub["PC1"], sub["PC2"],
        s=38,
        c=COLORS[group],
        alpha=0.75,
        marker="o",
        edgecolors="black",
        linewidths=0.4,
        label=group,
        zorder=2
    )
seen = set()
for behar_name, (base_group, label_name) in BEHAR_MAP.items():
    sub = df[df["population"] == behar_name]
    if len(sub) == 0:
        continue
    label = label_name if label_name not in seen else None
    seen.add(label_name)
    ax1.scatter(
        sub["PC1"], sub["PC2"],
        s=70,
        c=COLORS[base_group],
        alpha=0.8,
        marker="x",
        linewidths=1.6,
        label=label,
        zorder=3
    )
ax1.set_title("A)", loc="left", fontsize=16)
ax1.grid(False)
ax1.tick_params(axis="both", labelsize=12)
core_zoom = core_jewish[
    core_jewish["plot_group"] != "Uzbekistani Jews (Behar)"
].copy()
for group in MY_GROUPS:
    sub = core_zoom[core_zoom["plot_group"] == group]
    if len(sub) == 0:
        continue
    ax2.scatter(
        sub["PC1"], sub["PC2"],
        s=46,
        c=COLORS[group],
        alpha=0.8,
        marker="o",
        edgecolors="black",
        linewidths=0.4,
        zorder=2
    )
for behar_name, (base_group, _) in BEHAR_MAP.items():
    sub = core_zoom[core_zoom["population"] == behar_name]
    if len(sub) == 0:
        continue
    ax2.scatter(
        sub["PC1"], sub["PC2"],
        s=56,
        c=COLORS[base_group],
        alpha=0.8,
        marker="x",
        linewidths=1.5,
        zorder=3
    )
x_min = core_zoom["PC1"].quantile(0.01)
x_max = core_zoom["PC1"].quantile(0.99)
y_min = core_zoom["PC2"].quantile(0.01)
y_max = core_zoom["PC2"].quantile(0.99)
pad_x = (x_max - x_min) * 0.12
pad_y = (y_max - y_min) * 0.12
ax2.set_xlim(x_min - pad_x, x_max + pad_x)
ax2.set_ylim(y_min - pad_y, y_max + pad_y)
ax2.set_title("B)", loc="left", fontsize=16)
ax2.grid(False)
ax2.tick_params(axis="both", labelsize=12)
if pc1_var is not None:
    xlab = f"PC1 ({pc1_var:.2f}%)"
else:
    xlab = "PC1"
if pc2_var is not None:
    ylab = f"PC2 ({pc2_var:.2f}%)"
else:
    ylab = "PC2"
ax1.set_xlabel(xlab, fontsize=15)
ax1.set_ylabel(ylab, fontsize=15)
ax2.set_xlabel(xlab, fontsize=15)
ax2.set_ylabel(ylab, fontsize=15)
handles, labels = ax1.get_legend_handles_labels()
uniq = {}
for h, l in zip(handles, labels):
    if l and l not in uniq:
        uniq[l] = h
legend_order = [
    "Ashkenazi Jews",
    "Ashkenazi Jews (Behar)",
    "Bukharan Jews",
    "Uzbekistani Jews (Behar)",
    "Georgian Jews",
    "Georgian Jews (Behar)",
    "Kurdistan Jews",
    "Iraqi Jews (Behar)",
    "Mountain Jews",
    "Azerbaijani Jews (Behar)",
    "Iranian Jews (Behar)",
    "Non-Jewish",
]
ordered_labels = [lab for lab in legend_order if lab in uniq]
ordered_handles = [uniq[lab] for lab in ordered_labels]
fig.legend(
    ordered_handles,
    ordered_labels,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.02),
    ncol=6,
    frameon=True,
    facecolor="white",
    edgecolor="black",
    fontsize=12
)
fig.subplots_adjust(
    left=0.08,
    right=0.98,
    top=0.90,
    bottom=0.15,
    wspace=0.20
)
fig.savefig(OUT_FIG, dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
EIGENVEC = "pca_common_merged_dedup.eigenvec"
EIGENVAL = "pca_common_merged_dedup.eigenval"
POP_FILE = "pop_merged.tsv"
OUT_FIG = "PCAwBehar_2_txt.png"
eig = pd.read_csv(EIGENVEC, sep=r"\s+")
cols = list(eig.columns)
cols[0] = "FID"
cols[1] = "IID"
eig.columns = cols
pc_names = [c for c in eig.columns if str(c).startswith("PC")]
if len(pc_names) < 2:
    raise ValueError("PC columns not found in eigenvec")
eig = eig[["IID"] + pc_names].copy()
for c in pc_names:
    eig[c] = pd.to_numeric(eig[c], errors="coerce")
eig["IID"] = eig["IID"].astype(str).str.strip()
eig = eig.dropna(subset=[pc_names[0], pc_names[1]]).copy()
eig = eig.rename(columns={pc_names[0]: "PC1", pc_names[1]: "PC2"})
evals = pd.read_csv(EIGENVAL, header=None)
evals = pd.to_numeric(evals.iloc[:, 0], errors="coerce").dropna()
var_exp = evals / evals.sum() * 100
pc1_var = float(var_exp.iloc[0]) if len(var_exp) > 0 else None
pc2_var = float(var_exp.iloc[1]) if len(var_exp) > 1 else None
pop = pd.read_csv(POP_FILE, sep="\t")
if list(pop.columns[:2]) != ["sample_id", "population"]:
    pop = pop.iloc[:, :2].copy()
    pop.columns = ["sample_id", "population"]
pop["sample_id"] = pop["sample_id"].astype(str).str.strip()
pop["population"] = pop["population"].astype(str).str.strip()
df = eig.merge(pop, left_on="IID", right_on="sample_id", how="left")
df = df[df["population"].notna()].copy()
df = df[~df["population"].isin(["Unknown", "No Tag", "Unknown_Behar"])].copy()
COLORS = {
    "Ashkenazi Jews": "#7b2cbf",
    "Bukharan Jews": "#6ec5ff",
    "Georgian Jews": "#d62728",
    "Kurdistan Jews": "#8fd175",
    "Mountain Jews": "#f39c12",
    "Iranian Jews": "#1f77b4",
    "Non-Jewish": "#8f8f8f"
}
BEHAR_MAP = {
    "Ashkenazi_Jews_Behar": ("Ashkenazi Jews", "Ashkenazi Jews (Behar)"),
    "Uzbekistan_Jews_Behar": ("Bukharan Jews", "Uzbekistan Jews (Behar)"),
    "Georgian_Jews_Behar": ("Georgian Jews", "Georgian Jews (Behar)"),
    "Iraqi_Jews_Behar": ("Kurdistan Jews", "Iraqi Jews (Behar)"),
    "Azerbaijani_Jews_Behar": ("Mountain Jews", "Azerbaijani Jews (Behar)"),
    "Mountain_Jews_Behar": ("Mountain Jews", "Azerbaijani Jews (Behar)"),
    "Iranian_Jews_Behar": ("Iranian Jews", "Iranian Jews (Behar)"),
}
MY_GROUPS = [
    "Ashkenazi Jews",
    "Bukharan Jews",
    "Georgian Jews",
    "Kurdistan Jews",
    "Mountain Jews",
]
REF_GROUP_ORDER = [
    "Ashkenazi Jews (Behar)",
    "Uzbekistan Jews (Behar)",
    "Georgian Jews (Behar)",
    "Iraqi Jews (Behar)",
    "Azerbaijani Jews (Behar)",
    "Iranian Jews (Behar)",
]
def get_plot_group(pop_name):
    if pop_name in MY_GROUPS:
        return pop_name
    if pop_name in BEHAR_MAP:
        return BEHAR_MAP[pop_name][1]
    return "Non-Jewish"
df["plot_group"] = df["population"].apply(get_plot_group)
keep_groups = MY_GROUPS + REF_GROUP_ORDER + ["Non-Jewish"]
df = df[df["plot_group"].isin(keep_groups)].copy()
core_jewish = df[df["plot_group"] != "Non-Jewish"].copy()
other = df[df["plot_group"] == "Non-Jewish"].copy()
fig, (ax1, ax2) = plt.subplots(
    2, 1,
    figsize=(9, 14),
    dpi=300,
    gridspec_kw={"height_ratios": [1.15, 1]}
)
if len(other) > 0:
    ax1.scatter(
        other["PC1"], other["PC2"],
        s=14,
        c=COLORS["Non-Jewish"],
        alpha=0.5,
        marker="o",
        edgecolors="none",
        label="Non-Jewish",
        zorder=1
    )
for group in MY_GROUPS:
    sub = df[df["plot_group"] == group]
    if len(sub) == 0:
        continue
    ax1.scatter(
        sub["PC1"], sub["PC2"],
        s=38,
        c=COLORS[group],
        alpha=0.75,
        marker="o",
        edgecolors="black",
        linewidths=0.4,
        label=group,
        zorder=2
    )
seen = set()
for behar_name, (base_group, label_name) in BEHAR_MAP.items():
    sub = df[df["population"] == behar_name]
    if len(sub) == 0:
        continue
    label = label_name if label_name not in seen else None
    seen.add(label_name)
    ax1.scatter(
        sub["PC1"], sub["PC2"],
        s=70,
        c=COLORS[base_group],
        alpha=0.8,
        marker="x",
        linewidths=1.6,
        label=label,
        zorder=3
    )
ax1.set_title("A)", loc="left", fontsize=16)
ax1.grid(False)
ax1.tick_params(axis="both", labelsize=12)
core_zoom = df[df["plot_group"] != "Non-Jewish"].copy()
for group in MY_GROUPS:
    sub = core_zoom[core_zoom["plot_group"] == group]
    if len(sub) == 0:
        continue
    ax2.scatter(
        sub["PC1"], sub["PC2"],
        s=46,
        c=COLORS[group],
        alpha=0.8,
        marker="o",
        edgecolors="black",
        linewidths=0.4,
        zorder=2
    )
for behar_name, (base_group, _) in BEHAR_MAP.items():
    sub = core_zoom[core_zoom["population"] == behar_name]
    if len(sub) == 0:
        continue
    ax2.scatter(
        sub["PC1"], sub["PC2"],
        s=56,
        c=COLORS[base_group],
        alpha=0.8,
        marker="x",
        linewidths=1.5,
        zorder=3
    )
x_min = core_zoom["PC1"].quantile(0.01)
x_max = core_zoom["PC1"].quantile(0.99)
y_min = core_zoom["PC2"].quantile(0.01)
y_max = core_zoom["PC2"].quantile(0.99)
pad_x = (x_max - x_min) * 0.12
pad_y = (y_max - y_min) * 0.12
ax2.set_xlim(x_min - pad_x, x_max + pad_x)
ax2.set_ylim(y_min - pad_y, y_max + pad_y)
ax2.set_title("B)", loc="left", fontsize=16)
ax2.grid(False)
ax2.tick_params(axis="both", labelsize=12)
if pc1_var is not None:
    xlab = f"PC1 ({pc1_var:.2f}%)"
else:
    xlab = "PC1"
if pc2_var is not None:
    ylab = f"PC2 ({pc2_var:.2f}%)"
else:
    ylab = "PC2"
ax1.set_xlabel(xlab, fontsize=15)
ax1.set_ylabel(ylab, fontsize=15)
ax2.set_xlabel(xlab, fontsize=15)
ax2.set_ylabel(ylab, fontsize=15)
handles, labels = ax1.get_legend_handles_labels()
uniq = {}
for h, l in zip(handles, labels):
    if l and l not in uniq:
        uniq[l] = h
legend_order = [
    "Ashkenazi Jews",
    "Ashkenazi Jews (Behar)",
    "Bukharan Jews",
    "Uzbekistan Jews (Behar)",
    "Georgian Jews",
    "Georgian Jews (Behar)",
    "Kurdistan Jews",
    "Iraqi Jews (Behar)",
    "Mountain Jews",
    "Azerbaijani Jews (Behar)",
    "Iranian Jews (Behar)",
    "Non-Jewish",
]
ordered_labels = [lab for lab in legend_order if lab in uniq]
ordered_handles = [uniq[lab] for lab in ordered_labels]
fig.legend(
    ordered_handles,
    ordered_labels,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.02),
    ncol=3,
    frameon=True,
    facecolor="white",
    edgecolor="black",
    fontsize=10,
    markerscale=0.8,
    columnspacing=0.8,
    handletextpad=0.4,
)
fig.subplots_adjust(
    left=0.10,
    right=0.98,
    top=0.96,
    bottom=0.087,
    hspace=0.22
)
fig.savefig(OUT_FIG, dpi=300, bbox_inches="tight")
plt.show()
